# Look up instances by video id

Every labelled WLASL-2000 instance (the other splits are subsets of it), keyed by `video_id`: its
gloss, set, signer and source, and its clip at its real length and speed. For browsing a whole
class, see [class_viewer.ipynb](./class_viewer.ipynb). Clips show the labelled frame range: for an
instance whose annotated range was unusable, that is the whole clip rather than WLASL's annotation
(see [frame-range resets](../../info/WLASL_info.md#frame-range-resets)).

In [ ]:
import re
from collections.abc import Iterable

import pandas as pd
from IPython.display import display

from src.preprocess import Instance
from src.video_dataset import load_split_instances
from src.visualise2 import (
    animate_clip_grid,
    animate_frames,
    animation_html,
    load_instance_frames,
    set_thesis_style,
)

set_thesis_style()
instances = load_split_instances("asl2000")


def parse_ids(text: str) -> list[str]:
    """The 5-digit video ids in `text`, in order, e.g. a cell of `duplicate_clips.ipynb`'s
    `video_ids` column (`01466, 40565, 47305`)."""
    return re.findall(r"\b\d{5}\b", text)


def look_up(video_ids: Iterable[str]) -> list[Instance]:
    """The instance of each video id, raising on any that isn't labelled."""
    ids = list(video_ids)
    missing = [v for v in ids if v not in instances]
    if missing:
        raise KeyError(f"Not labelled in asl2000: {missing}")
    return [instances[v] for v in ids]


def describe(found: list[Instance]) -> pd.DataFrame:
    """One row of metadata per instance."""
    return pd.DataFrame(
        [
            {
                "video_id": i.video_id,
                "gloss": i.label_name,
                "set": i.split,
                "signer": i.signer_id,
                "variation": i.variation_id,
                "source": i.source,
                "frames": i.frame_end - i.frame_start,
                "url": i.url,
            }
            for i in found
        ]
    ).set_index("video_id")


def show_metadata(found: list[Instance]) -> None:
    """Display `describe(found)`, with URLs in full."""
    with pd.option_context("display.max_colwidth", None):
        display(describe(found))


len(instances)

## One video

In [ ]:
video_id = "05741"  # change to look up another video

(instance,) = look_up([video_id])
show_metadata([instance])
title = f"{instance.video_id}: {instance.label_name}"
display(animation_html(*animate_frames(load_instance_frames(instance), title=title)))

## Several videos in a grid

Give `video_ids` as a list, or paste text containing them into `parse_ids`. The clips play in
step; shorter ones hold their last frame.

In [ ]:
video_ids = parse_ids("17016, 42541, 17017, 42542")  # or a list: ["05741", "41452"]
cols = 4

found = look_up(video_ids)
show_metadata(found)
titles = [
    f"{i.video_id}: {i.label_name}\n{i.split}, signer {i.signer_id}" for i in found
]
clips = [load_instance_frames(i) for i in found]
display(animation_html(*animate_clip_grid(clips, cols, titles=titles)))